<a href="https://colab.research.google.com/github/diesalgueiros-wq/E-Commerce-Predictive-Analytics/blob/main/notebooks/01_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import io

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/TFM_IA_DiegoSalgueiro/data /Online Retail.xlsx'

df_raw = pd.read_excel(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_clean = df_raw.dropna(subset=['CustomerID']).copy()
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0] # Aseguramos precios válidos
df_clean['TotalSum'] = df_clean['Quantity'] * df_clean['UnitPrice']

customers = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (df_clean['InvoiceDate'].max() - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalSum': 'sum',
    'StockCode': 'nunique'
}).reset_index()

customers.columns = ['CustomerID', 'Recency', 'Frequency', 'MonetaryValue', 'ProductVariety']


q1 = customers['MonetaryValue'].quantile(0.25)
q3 = customers['MonetaryValue'].quantile(0.75)
ric = q3 - q1
mask = (customers['MonetaryValue'] >= q1 - 1.5*ric) & (customers['MonetaryValue'] <= q3 + 1.5*ric)
customers = customers[mask].copy()

limite_frecuencia = customers['Frequency'].quantile(0.80)
limite_gasto = customers['MonetaryValue'].quantile(0.80)

customers['Is_VIP'] = ((customers['Frequency'] > limite_frecuencia) &
                        (customers['MonetaryValue'] > limite_gasto)).astype(int)

final_table = customers.drop(["CustomerID"], axis=1).reset_index(drop=True)

print("Dataset Preprocesado Real (Sin datos sintéticos):")
display(final_table.head(10))

output_path = '/content/drive/MyDrive/TFM_IA_DiegoSalgueiro/Online_Retail_Processed.csv'
customers.to_csv(output_path, index=False) # index=False evita la columna 'Unnamed: 0'

print(f"\n ¡El dataset procesado ha sido guardado con éxito en: {output_path}")

Dataset Preprocesado Real (Sin datos sintéticos):


,Recency,Frequency,MonetaryValue,ProductVariety,Is_VIP
0,74,4,1797.24,22,0
1,18,1,1757.55,73,0
2,309,1,334.40,17,0
3,35,8,2506.04,59,1
4,203,1,89.00,4,0
5,231,1,1079.40,58,0
6,213,1,459.40,13,0
7,22,3,2811.43,53,0
8,1,2,1168.06,13,0
9,51,3,2662.06,105,0



 ¡El dataset procesado ha sido guardado con éxito en: /content/drive/MyDrive/TFM_IA_DiegoSalgueiro/Online_Retail_Processed.csv


In [ ]:
output_path = '/content/drive/MyDrive/Online_Retail_Processed.xlsx'

# Export to Excel
customers.to_excel(output_path, index=True)

print(f"The processed dataset has been saved to: {output_path}")

The processed dataset has been saved to: /content/drive/MyDrive/Online_Retail_Processed.xlsx
